# 01a — Rule-Based Systems vs Learning from Data

This notebook demonstrates the core difference between **traditional programming** and **machine learning** using pure Python — no libraries.

We build the same task twice:
1. A rule-based system (expert system style)
2. A system that learns the rule from data

**Task:** Predict whether a student passes or fails based on their exam score.

---
## Part 1: The Rule-Based Approach

A human expert writes the rule explicitly.
The program applies it.

In [1]:
# Traditional programming: human writes the rule
# Rule: if score >= 50, pass. Otherwise, fail.

def predict_rule_based(score):
    if score >= 50:
        return 'pass'
    else:
        return 'fail'

# Test it
test_scores = [30, 45, 50, 65, 80]

print('Rule-Based Predictions')
print('-' * 30)
for score in test_scores:
    print(f'  Score: {score}  →  {predict_rule_based(score)}')

Rule-Based Predictions
------------------------------
  Score: 30  →  fail
  Score: 45  →  fail
  Score: 50  →  pass
  Score: 65  →  pass
  Score: 80  →  pass


This works — but only because **we already knew the rule** (pass mark = 50).

What if the pass mark is different for different courses? What if it depends on the cohort average? What if there are multiple features (attendance, coursework, exam)?

The rules multiply. They become hard to maintain. And crucially — **we had to know the answer before we wrote the program**.

---
## Part 2: The Knowledge Bottleneck

Let's make the problem slightly harder — two features instead of one.
Now the rule is not obvious.

In [2]:
# Two features: exam score and attendance percentage
# Label: 1 = pass, 0 = fail

# Training data: (exam_score, attendance, label)
data = [
    (80, 90, 1),
    (75, 85, 1),
    (60, 70, 1),
    (55, 60, 1),
    (40, 80, 0),
    (30, 50, 0),
    (70, 30, 0),   # high score but very low attendance — fails
    (50, 40, 0),
]

print('Training Data')
print('-' * 45)
print(f'  {"Exam":>6}  {"Attend":>6}  {"Label":>6}')
for exam, attend, label in data:
    result = 'pass' if label == 1 else 'fail'
    print(f'  {exam:>6}  {attend:>6}  {result:>6}')

Training Data
---------------------------------------------
    Exam  Attend   Label
      80      90    pass
      75      85    pass
      60      70    pass
      55      60    pass
      40      80    fail
      30      50    fail
      70      30    fail
      50      40    fail


In [3]:
# Try to write a rule by hand
# Notice: sample (70, 30) fails despite high exam score
# The rule must account for both features

def predict_manual_rule(exam, attend):
    if exam >= 55 and attend >= 60:
        return 1
    else:
        return 0

print('Manual Rule Predictions vs True Labels')
print('-' * 50)
correct = 0
for exam, attend, true_label in data:
    pred = predict_manual_rule(exam, attend)
    match = '✓' if pred == true_label else '✗'
    print(f'  exam={exam}, attend={attend}  →  pred={pred}, true={true_label}  {match}')
    if pred == true_label:
        correct += 1

print(f'\nAccuracy: {correct}/{len(data)} = {correct/len(data)*100:.0f}%')
print('\nThe rule was written by hand — it may not be optimal.')
print('With more features, writing rules by hand becomes impossible.')

Manual Rule Predictions vs True Labels
--------------------------------------------------
  exam=80, attend=90  →  pred=1, true=1  ✓
  exam=75, attend=85  →  pred=1, true=1  ✓
  exam=60, attend=70  →  pred=1, true=1  ✓
  exam=55, attend=60  →  pred=1, true=1  ✓
  exam=40, attend=80  →  pred=0, true=0  ✓
  exam=30, attend=50  →  pred=0, true=0  ✓
  exam=70, attend=30  →  pred=0, true=0  ✓
  exam=50, attend=40  →  pred=0, true=0  ✓

Accuracy: 8/8 = 100%

The rule was written by hand — it may not be optimal.
With more features, writing rules by hand becomes impossible.


---
## Part 3: The ML Approach — Learning the Rule from Data

Instead of writing the rule, we let the system **find it from the data**.

We define a simple model with two weights (one per feature) and a bias.
The model adjusts these weights to minimise prediction error.

This is the core idea of machine learning — no libraries, pure Python.

In [4]:
# Simple linear model: score = w1*exam + w2*attend + b
# Predict 1 (pass) if score >= 0, else 0 (fail)

# Normalise features to [0, 1] range so weights are comparable
def normalise(value, min_val, max_val):
    return (value - min_val) / (max_val - min_val)

# Prepare normalised data
normalised_data = [
    (normalise(exam, 30, 80), normalise(attend, 30, 90), label)
    for exam, attend, label in data
]

print('Normalised Training Data')
print('-' * 45)
print(f'  {"Exam":>8}  {"Attend":>8}  {"Label":>6}')
for exam_n, attend_n, label in normalised_data:
    print(f'  {exam_n:>8.3f}  {attend_n:>8.3f}  {label:>6}')

Normalised Training Data
---------------------------------------------
      Exam    Attend   Label
     1.000     1.000       1
     0.900     0.917       1
     0.600     0.667       1
     0.500     0.500       1
     0.200     0.833       0
     0.000     0.333       0
     0.800     0.000       0
     0.400     0.167       0


In [5]:
# Initialise weights and bias
w1 = 0.0   # weight for exam score
w2 = 0.0   # weight for attendance
b  = 0.0   # bias
lr = 0.1   # learning rate

def predict(exam_n, attend_n, w1, w2, b):
    z = w1 * exam_n + w2 * attend_n + b
    return 1 if z >= 0 else 0

# Training loop — perceptron-style update
print(f'Initial weights: w1={w1}, w2={w2}, b={b}')
print()

for epoch in range(1, 21):
    errors = 0
    for exam_n, attend_n, true_label in normalised_data:
        pred = predict(exam_n, attend_n, w1, w2, b)
        error = true_label - pred
        if error != 0:
            w1 += lr * error * exam_n
            w2 += lr * error * attend_n
            b  += lr * error
            errors += 1
    if epoch <= 5 or errors == 0:
        print(f'Epoch {epoch:>2}: errors={errors}  w1={w1:.4f}  w2={w2:.4f}  b={b:.4f}')
    if errors == 0:
        print(f'\nConverged at epoch {epoch} — no more errors.')
        break

Initial weights: w1=0.0, w2=0.0, b=0.0

Epoch  1: errors=1  w1=-0.0200  w2=-0.0833  b=-0.1000
Epoch  2: errors=2  w1=0.0600  w2=-0.0667  b=-0.1000
Epoch  3: errors=3  w1=0.0600  w2=-0.0500  b=-0.2000
Epoch  4: errors=2  w1=0.0800  w2=0.0500  b=-0.2000
Epoch  5: errors=2  w1=0.1600  w2=0.0667  b=-0.2000


In [6]:
# Evaluate learned model
print('Learned Model Predictions vs True Labels')
print('-' * 55)
correct = 0
for (exam_n, attend_n, true_label), (exam, attend, _) in zip(normalised_data, data):
    pred = predict(exam_n, attend_n, w1, w2, b)
    match = '✓' if pred == true_label else '✗'
    print(f'  exam={exam}, attend={attend}  →  pred={pred}, true={true_label}  {match}')
    if pred == true_label:
        correct += 1

print(f'\nAccuracy: {correct}/{len(data)} = {correct/len(data)*100:.0f}%')
print(f'\nLearned weights: w1={w1:.4f} (exam), w2={w2:.4f} (attend), b={b:.4f}')
print('\nThe model found the rule from data — we never wrote it explicitly.')

Learned Model Predictions vs True Labels
-------------------------------------------------------
  exam=80, attend=90  →  pred=1, true=1  ✓
  exam=75, attend=85  →  pred=1, true=1  ✓
  exam=60, attend=70  →  pred=0, true=1  ✗
  exam=55, attend=60  →  pred=0, true=1  ✗
  exam=40, attend=80  →  pred=0, true=0  ✓
  exam=30, attend=50  →  pred=0, true=0  ✓
  exam=70, attend=30  →  pred=0, true=0  ✓
  exam=50, attend=40  →  pred=0, true=0  ✓

Accuracy: 6/8 = 75%

Learned weights: w1=0.4300 (exam), w2=0.2083 (attend), b=-0.4000

The model found the rule from data — we never wrote it explicitly.


---
## Part 4: What Just Happened

Compare the two approaches:

| | Rule-Based | ML |
|---|---|---|
| Who writes the rule? | Human | Nobody — it emerges from data |
| What if features change? | Rewrite rules | Retrain on new data |
| Scales to 100 features? | No | Yes |
| Requires knowing the answer first? | Yes | No |

The weights `w1` and `w2` that the model learned encode the relative importance of exam score vs attendance. We did not decide this — the data did.

In [7]:
# Predict on a new, unseen student
new_exam    = 65
new_attend  = 55

new_exam_n   = normalise(new_exam,   30, 80)
new_attend_n = normalise(new_attend, 30, 90)

pred = predict(new_exam_n, new_attend_n, w1, w2, b)
result = 'pass' if pred == 1 else 'fail'

print(f'New student: exam={new_exam}, attendance={new_attend}%')
print(f'Prediction: {result}')
print()
print('The model generalises to students it has never seen.')

New student: exam=65, attendance=55%
Prediction: fail

The model generalises to students it has never seen.
